In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType

In [0]:
df = spark.table("workspace.bronze.category_translation")


In [0]:
string_cols = [f.name for f in df.schema.fields if isinstance(f.dataType, StringType)]

df = df.select(
    *[F.trim(F.col(c)).alias(c) if c in string_cols else F.col(c) for c in df.columns]
)

In [0]:
# 3. TRANSFORM & NORMALIZE
df = (
    df
    .withColumnRenamed("product_category_name", "category_name_portuguese")
    .withColumnRenamed("product_category_name_english", "category_name_english")
    
    .filter(F.col("category_name_portuguese").isNotNull())
    
    .dropDuplicates(["category_name_portuguese"])
    
    # Portuguese Cleaning (keep underscore for joins)
    .withColumn(
        "category_name_portuguese",
        F.lower(
            F.translate(
                F.col("category_name_portuguese"),
                "áàâãéèêíìîóòôõúùûçÁÀÂÃÉÈÊÍÌÎÓÒÔÕÚÙÛÇ",
                "aaaaeeeiiioooouuucAAAAEEEIIIOOOOUUUC"
            )
        )
    )
    
    # English Cleaning (for dashboards)
    .withColumn(
        "category_name_english",
        F.initcap(
            F.regexp_replace(
                F.coalesce(F.col("category_name_english"), F.lit("unknown")),
                "_",
                " "
            )
        )
    )
    
    .withColumn("silver_processed_time", F.current_timestamp())
)

In [0]:
df.show()

In [0]:
(
    df.write
    .mode("overwrite")
    .format("delta")
    .option("overwriteSchema", "true")
    .saveAsTable("workspace.silver.category_translation")
)

